In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')

In [3]:
df_movies = pd.read_csv('tmdb_5000_movies.csv')
df_movies.info()

<class 'pandas.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   str    
 2   homepage              1712 non-null   str    
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   str    
 5   original_language     4803 non-null   str    
 6   original_title        4803 non-null   str    
 7   overview              4800 non-null   str    
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   str    
 10  production_countries  4803 non-null   str    
 11  release_date          4802 non-null   str    
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   str    
 15  status                4803 non-n

In [4]:
movies_analysis = df_movies[
    [
        "id",
        "budget",
        "genres",
        "title",
        "release_date",
        "revenue",
        "vote_average",
        "vote_count",
    ]
].copy()

In [5]:
def quality_report(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''데이터프레임의 품질을 컬럼별로 진단해 한 표로 돌려줍니다.'''
    n_rows = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
        "sample": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
    })
    print(f"[품질 리포트] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  메모리: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return report

qr_movies = quality_report(df_movies, "df_movies")
display(qr_movies)

[품질 리포트] df_movies
  행 수: 4,803  /  열 수: 20
  메모리: 5542.1 KB
  완전 중복 행: 0건


,dtype,missing,missing_pct,n_unique,sample
budget,int64,0,0.00,436,237000000
genres,str,0,0.00,1175,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
homepage,str,3091,64.36,1691,http://www.avatarmovie.com/
id,int64,0,0.00,4803,19995
keywords,str,0,0.00,4222,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":..."
original_language,str,0,0.00,37,en
original_title,str,0,0.00,4801,Avatar
overview,str,3,0.06,4800,"In the 22nd century, a paraplegic Marine is di..."
popularity,float64,0,0.00,4802,150.437577
production_companies,str,0,0.00,3697,"[{""name"": ""Ingenious Film Partners"", ""id"": 289..."


In [6]:
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base

qr_movies_full = quality_report_full(df_movies, "df_movies")
display(qr_movies_full)

[품질 리포트(완전판)] df_movies
  행 수: 4,803  /  열 수: 20
  완전 중복 행: 0건
  📌 날짜로 보이는 object 컬럼: ['release_date']


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
budget,int64,0,0.00,436,6.68,False
genres,str,0,0.00,1175,NaN,False
homepage,str,3091,64.36,1691,NaN,False
id,int64,0,0.00,4803,14.66,False
keywords,str,0,0.00,4222,NaN,False
original_language,str,0,0.00,37,NaN,False
original_title,str,0,0.00,4801,NaN,False
overview,str,3,0.06,4800,NaN,False
popularity,float64,0,0.00,4802,5.73,False
production_companies,str,0,0.00,3697,NaN,False


In [7]:
def drop_duplicates_step(df):
    '''1단계: 완전 중복 행 제거.'''
    return df.drop_duplicates(keep="first").reset_index(drop=True)

def clean_channel_step(df):
    '''2단계: channel의 공백·대소문자 정제.'''
    return df.assign(channel=df["channel"].str.strip().str.lower())

def parse_date_step(df):
    '''3단계: order_date 문자열을 datetime으로 파싱.'''
    return df.assign(
        release_date=pd.to_datetime(
            df["release_date"],
            format="mixed",
            errors="coerce"))

def remove_outliers_step(df):
    '''4단계: 이상치 제거.'''
    Q3_budget = df['budget'].quantile(0.75)     
    Q1_budget = df['budget'].quantile(0.25)     
    IQR_budget = Q3_budget - Q1_budget

    Q3_revenue = df['revenue'].quantile(0.75)
    Q1_revenue = df['revenue'].quantile(0.25)
    IQR_revenue = Q3_revenue - Q1_revenue

    upper_boundary_budget = Q3_budget + 1.5 * IQR_budget
    lower_boundary_budget = Q1_budget - 1.5 * IQR_budget

    upper_boundary_revenue = Q3_revenue + 1.5 * IQR_revenue
    lower_boundary_revenue = Q1_revenue - 1.5 * IQR_revenue

    return (
        df[df["budget"].between(lower_boundary_budget, upper_boundary_budget)
               & df["revenue"].between(lower_boundary_revenue, upper_boundary_revenue)])

def drop_missing_step(df):
    '''5단계: NaN값 제거.'''
    return df.dropna().reset_index(drop=True)

In [8]:
movies_clean = (
    movies_analysis
    .pipe(drop_duplicates_step)
    .pipe(parse_date_step)
    .pipe(remove_outliers_step)
    .pipe(drop_missing_step)
)

print(f"정제 전: {df_movies.shape} → 정제 후: {movies_clean.shape}")

정제 전: (4803, 20) → 정제 후: (4250, 8)


In [9]:
import ast

movies_clean["year"] = movies_clean["release_date"].dt.year
movies_clean["month"] = movies_clean["release_date"].dt.month

# KPI 1: 개봉 연도별·월별 총수익 합계(wide)
year_month_revenue = movies_clean.groupby(["year", "month"])["revenue"].sum().unstack(fill_value=0).round(0)

print("개봉 연도별·월별 총수익:")
display(year_month_revenue)

# genres 문자열에서 첫 번째 장르 추출
def getGenre(gnr):
    gnr = ast.literal_eval(gnr) if isinstance(gnr, str) else gnr
    return gnr[0]["name"] if gnr else "Unknown"

movies_clean["genre"] = movies_clean["genres"].apply(getGenre)

# KPI 2: 장르별 평균 평점·총수익·영화 수
genre_kpi = (
    movies_clean.groupby("genre")
    .agg(avg_rating=("vote_average", "mean"), total_revenue=("revenue", "sum"), n_movies=("id", "count"))
    .round({"avg_rating": 2, "total_revenue": 0})
    .sort_values("total_revenue", ascending=False)
)

print("\n장르별 KPI:")
display(genre_kpi)

# 각 KPI의 1위 확인
top_year_month = movies_clean.groupby(["year", "month"])["revenue"].sum().nlargest(1)
top_genre = genre_kpi.head(1)

print("\n총수익이 가장 큰 연도·월:")
display(top_year_month)

print("\n총수익이 가장 큰 장르:")
display(top_genre)

개봉 연도별·월별 총수익:


month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
1916,0,0,0,0,0,0,0,0,8394751,0,0,0
1925,0,0,0,0,0,0,0,0,0,0,22000000,0
1927,650422,0,0,0,0,0,0,0,0,0,0,0
1929,0,4358000,0,0,0,0,0,0,0,0,0,0
1930,0,0,0,0,0,0,0,0,0,0,8000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2013,923769954,541301566,464250488,393293019,299795029,370576181,424892289,706981492,680124995,1073510662,411486561,889393338
2014,874419893,408932302,317580298,412814509,165245090,418743018,519128327,1015450519,568963300,986598283,401279055,931989471
2015,233825738,494745644,316363980,375329032,78117333,560502602,699232812,613378135,870924754,534044529,439881595,552744721



장르별 KPI:


,avg_rating,total_revenue,n_movies
genre,,,
Comedy,5.91,36315105553,987
Drama,6.40,33658932131,1144
Action,5.85,28229738580,608
Adventure,6.14,12645462807,216
Horror,5.61,11733197018,295
Crime,6.40,8120764026,191
Thriller,5.57,6301402088,177
Fantasy,6.09,5650708894,88
Romance,6.14,4014162283,98



총수익이 가장 큰 연도·월:


year  month
2005  9        1697879568
Name: revenue, dtype: int64


총수익이 가장 큰 장르:


,avg_rating,total_revenue,n_movies
genre,,,
Comedy,5.91,36315105553,987


In [10]:
# Parquet 저장

from pathlib import Path

OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

year_month_revenue.to_parquet(OUT_DIR / "movie_year_month_revenue.parquet")
genre_kpi.to_parquet(OUT_DIR / "movie_genre_kpi.parquet")

print(f"Parquet 저장 완료: {OUT_DIR.resolve()}")

Parquet 저장 완료: C:\git\ai-data-bootcamp\D009\output


# 영화 데이터 정제·집계 보고서

## 1. 데이터 개요
- 출처: TMDB 5000 Movies Dataset
- 기간: 1916-09-04 ~ 2017-02-03
- 분석 컬럼: id, budget, genres, title, release_date, revenue, vote_average, vote_count
- 원본 행 수 / 정제 후 행 수: 4,803행 → 4,250행

## 2. 발견한 품질 문제
- 결측: release_date 컬럼에서 1건의 결측값이 발견됨.
- 중복: 선택한 8개 컬럼 기준 완전 중복 행은 0건이었음.
- 이상치: budget과 revenue에서 IQR 범위를 벗어난 극단값이 확인됨.
- 데이터 형식: release_date는 object형이었으며, genres는 리스트 형태의 정보가 문자열로 저장되어 있었음.

## 3. 처리 결정과 근거
1. 분석에 필요한 8개 컬럼만 선택하여 분석과 관계없는 컬럼을 제외함.
2. 완전 중복 행을 점검하고 최초 행을 유지하도록 중복 제거 함수를 적용함.
3. release_date를 날짜 연산이 가능한 datetime형으로 변환함.
4. budget과 revenue의 극단적으로 큰 값 때문에 산점도에서 일반 영화의 분포가 압축되어 보여, IQR 기준을 벗어난 552개 행을 제외함.
5. release_date가 결측인 1개 행을 제거하여 최종적으로 4,250개 행을 분석에 사용함.

## 4. 주요 KPI 결과
- (연도·월 관점) 2005년 9월의 총수익이 1,697,879,568로 가장 높았음.
- (장르 관점) 첫 번째 등록 장르를 기준으로 Comedy의 총수익이 36,315,105,553으로 가장 높았음.

## 5. 한계와 후속 작업
- IQR 이상치는 오류 데이터가 아니라 실제 고예산 또는 고수익 영화일 수 있으므로, 이번 결과는 극단값을 제외한 영화들의 일반적인 분포를 나타냄.
- budget과 revenue가 0인 값은 실제 0인지 미등록 데이터인지 구분하기 어려워 추가 검토가 필요.
- 한 영화가 여러 장르를 가질 수 있지만 이번 집계에서는 첫 번째로 등록된 장르만 대표 장르로 사용함.

## 필수 과제

☑ quality_report_full() 함수를 빵셀러 데이터에 적용한 결과를 노트북에 포함했다.

☑ 정제 5단계를 작은 함수로 나누고 .pipe()로 묶었다.

☑ movies_clean을 Parquet으로 저장하고, 같은 데이터를 CSV로도 저장해 용량 차이를 한 줄로 적었다.

☑ 결정 로그 표(5단계 이상)를 채워 노트북에 박았다.

☑ 종합 프로젝트 보고서 5개 항목을 모두 채웠다.